# Publicación inicial Genie — ocho tablas export002

Preflight local por defecto. No SDK, credenciales, SQL, compute ni recursos en esa modalidad. La activación futura usa el perfil servidor del propietario existente del catálogo/esquema, no el lector app ni el Job writer. Coste desconocido. No existe espacio Genie creado por este notebook. Véase `skills/sbs-genie-datos/references/publication-writer.md` para recuperación, cuotas y límites.

In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd().resolve()
if not (ROOT / "src/sbs").exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from sbs.genie.publication_writer import load_write_plan, preflight, WriterConfig, sdk_writer
ACTIVATE = False
SERVER_CONFIG_PATH = None  # config pública del servidor; nunca contiene tokens
plan = load_write_plan(ROOT, publication_id="pilot002-016")
assert plan.payload == (ROOT / "runs/sk06-publication-writer-016-plan.json").read_bytes()
print(json.dumps(preflight(plan), indent=2))

In [ ]:
if ACTIVATE:
    if SERVER_CONFIG_PATH is None: raise ValueError("SERVER_CONFIGURATION_REQUIRED")
    server = json.loads(Path(SERVER_CONFIG_PATH).read_text())
    cfg = WriterConfig(**server["writer"])
    assert cfg.evidence_mode == "real"
    assert cfg.plan_sha256 == plan.sha256
    from databricks.sdk import Config
    # Autenticación normal del perfil instalador; nunca imprimir credenciales.
    auth = Config(profile=server["profile"])
    state = Path(server["state_root"]).resolve()
    with sdk_writer(plan, cfg, state / "journal", sdk_config=auth) as writer:
        result = writer.publish(certificate_directory=state / "certificates", registry_directory=state / "registry")
    print(json.dumps(result, indent=2))
else:
    print("PREFLIGHT_ONLY: 0 SDK instances, 0 remote SQL, 0 resource actions; publication pending.")